# EnergyCast V1 — Step 10: Linear Regression & Ridge Regularization
## Electricity Price Forecasting System (Indian Electricity Market)

**Objective:** Train and evaluate interpretable linear parametric baselines—Ordinary Least Squares (OLS) and L2-regularized Ridge Regression—across 53 leakage-free predictors.  
This notebook demonstrates feature scaling strictly on the training set, hyperparameter tuning of the regularization penalty $\alpha$ on the 2024 Validation set, out-of-sample evaluation on the 2025 Test set, and econometric coefficient interpretation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path

# Set visual style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)

# Paths
METRICS_PATH = Path('../outputs/tables/linear_ridge_metrics.csv')
COEF_PATH = Path('../outputs/tables/ridge_standardized_coefficients.csv')
TUNING_PATH = Path('../outputs/tables/ridge_alpha_tuning_results.csv')
PREDS_PATH = Path('../outputs/predictions/linear_ridge_test_predictions.csv')
print('Metrics CSV exists:', METRICS_PATH.exists())
print('Coefficients CSV exists:', COEF_PATH.exists())

### 1. Performance Comparison: Linear/Ridge vs Seasonal Naive
Comparing Ordinary Least Squares (OLS) and tuned Ridge ($\alpha=50.0$) against the Daily Seasonal Naive hurdle.

In [ ]:
metrics_df = pd.read_csv(METRICS_PATH)
display(metrics_df)

### 2. Ridge Regularization Path (Alpha Hyperparameter Tuning)
Evaluating validation MAE and RMSE across candidate regularization strengths $\alpha \in [0.01, 5000.0]$.

In [ ]:
tuning_df = pd.read_csv(TUNING_PATH)

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()
ax1.plot(tuning_df['Alpha'], tuning_df['Val_MAE'], marker='o', color='#1F4E79', lw=2, label='Val MAE')
ax2.plot(tuning_df['Alpha'], tuning_df['Val_RMSE'], marker='s', color='#D9534F', ls='--', lw=2, label='Val RMSE')
ax1.axvline(50.0, color='#E67E22', ls=':', lw=2, label='Optimal Alpha = 50.0')
ax1.set_xscale('log')
ax1.set_xlabel('Regularization Strength Alpha (Log Scale)')
ax1.set_ylabel('Validation MAE (Rs/MWh)', color='#1F4E79')
ax2.set_ylabel('Validation RMSE (Rs/MWh)', color='#D9534F')
ax1.set_title('Ridge Regularization Path on Validation Set (2024)', fontweight='bold')
plt.show()

### 3. Econometric Interpretation: Standardized Feature Weights (Coefficients)
Inspect the top positive and negative standardized coefficients ($eta_j$) to observe physical market drivers (e.g., negative weights on solar and wind; positive weights on net demand and peaker ramps).

In [ ]:
coef_df = pd.read_csv(COEF_PATH)
top_20 = coef_df.head(20).sort_values('Ridge_Coefficient', ascending=True)

plt.figure(figsize=(11, 7))
colors = ['#D9534F' if c < 0 else '#1F4E79' for c in top_20['Ridge_Coefficient']]
plt.barh(top_20['Feature'], top_20['Ridge_Coefficient'], color=colors, alpha=0.85)
plt.axvline(0, color='black', lw=0.8)
plt.title('Top 20 Standardized Ridge Coefficients (Econometric Feature Weights)', fontweight='bold')
plt.xlabel('Standardized Impact on MCP (Rs/MWh per 1σ feature change)')
plt.show()

### 4. Hourly MAE Comparison: Ridge vs Seasonal Naive
Comparing test set error profiles across the 24 hours of the day. Notice how Ridge substantially reduces the severe error peak during evening hours (18:00 to 22:00) compared to Seasonal Naive.

In [ ]:
test_preds = pd.read_csv(PREDS_PATH)
test_preds['timestamp'] = pd.to_datetime(test_preds['timestamp'])
snaive_preds = pd.read_csv('../outputs/predictions/snaive_test_predictions.csv')

test_preds['hour'] = test_preds['timestamp'].dt.hour
test_preds['err_ridge'] = np.abs(test_preds['mcp_rs_per_mwh'] - test_preds['pred_ridge'])
test_preds['err_snaive'] = np.abs(test_preds['mcp_rs_per_mwh'] - snaive_preds['pred_daily_snaive_24h'])

hourly_err = test_preds.groupby('hour')[['err_snaive', 'err_ridge']].mean()

plt.figure(figsize=(12, 5))
plt.plot(hourly_err.index, hourly_err['err_snaive'], marker='o', color='#E67E22', lw=2, label='Daily sNaive (24h) MAE')
plt.plot(hourly_err.index, hourly_err['err_ridge'], marker='^', color='#27AE60', lw=2.2, label='Ridge Regression (alpha=50) MAE')
plt.axvspan(18, 22, color='red', alpha=0.1, label='Critical Evening Peak (18:00 - 22:00)')
plt.title('Hourly MAE Profile on Test Set: Ridge Regression vs Seasonal Naive', fontweight='bold')
plt.xlabel('Hour of Day (0 to 23)')
plt.ylabel('Mean Absolute Error (Rs/MWh)')
plt.xticks(range(0, 24, 2))
plt.legend()
plt.show()

### 5. Summary & Hand-off to Step 11
**Ridge Regression ($\alpha=50.0$) has decisively beaten the Seasonal Naive hurdle**:
- **Test MAE:** ₹544.61 / MWh (vs sNaive ₹658.42 / MWh) — **17.3% error reduction**
- **Test RMSE:** ₹825.53 / MWh (vs sNaive ₹1,254.07 / MWh) — **34.2% error reduction**
- **Test R²:** **0.9059** (vs sNaive 0.7828)

We are now ready for **Step 11: Gradient Boosted Trees (LightGBM & XGBoost)** to capture non-linear interactions and piecewise regulatory price cap saturations.